# Карта зависимостей

## Часть 1 - Карта зависимостей StPlatform

### Классы-слои

- Контроллеры: `AssignmentPageController`, `CabinetController`, `TeacherAssignmentController`, `TeacherStudentController`, `TeacherImportController`, `ActivateController`, `PasswordResetController`, `LoginController`, `AttachmentController`, `AiSettingsPageController`, `ConsentPageController`; REST: `ApiSubmissionController`, `ApiAssignmentController`, `ApiAuthController`
- Сервисы: `UserService`, `AssignmentService`, `SubmissionService`, `ConsentService`, `ActivationService`, `PasswordResetService`, `StudentRemovalService`, `AiCheckService`, `AiSettingsService`, `PlatformUserDetailsService`
- Репозитории: `UserRepository`, `AssignmentRepository`, `SubmissionRepository`, `SubmissionAttachmentRepository`, `ConsentRecordRepository`, `AiCheckRepository`, `AiSettingsRepository`
- Сущности: `User`, `Assignment`, `Submission`, `SubmissionAttachment`, `ConsentRecord`, `AiCheck`, `AiSettings`

### Трассировка "ИИ-проверки сдачи"

Вместо «сдать домашку» выбрал ИИ-проверку сдачи: она интереснее, потому что идёт не одним запросом, а через очередь в БД и внешние API (LLM, GitHub/Gitverse). Заодно разобрасля, как платформа оценивает самостоятельность работы. Подозреваю, что «фраза «зaчем границa»» в критерии — ловушка для агента студента, который вставит её дословно :>

Пути от `src/main/java/ru/urfu/platform/`.

```
1. Постановка в очередь (запрос преподавателя):
AssignmentPageController.aiCheck (:121)
  → AiCheckService.isEnabled (:43)
  → AiCheckService.enqueueNew (:71) → AiCheckService.enqueue (:119)
      → SubmissionRepository.getReferenceById → AiCheckRepository.save        INSERT ai_checks, PENDING
   автоматический путь: SubmissionService.submit (:51) → SubmissionCreatedEvent (публикуется на :94)
      → AiCheckListener.onSubmissionCreated (:31, после коммита) → AiCheckService.enqueueIfAbsent (:59)

2. Фоновая проверка (поток планировщика, не HTTP-запрос):
AiCheckWorker.tick (:67, @Scheduled)
  → AiCheckService.claimNext (:141) → AiCheckService.claimOldestPending (:127)
      → AiCheckRepository.findFirstByStatusOrderByCreatedAtAsc                 PENDING → RUNNING
      → Submission, Assignment, SubmissionAttachmentRepository.findBySubmissionId → ReviewTask
  → AiCheckWorker.runOne (:84)
      → AttachmentStorage.pathOf → AttachmentTextExtractor.extract              текст вложения
      → AiReviewAgent.review (:66) → AiModelFactory.modelFor → ChatModel.call   запрос к LLM
            ⇄ RepoTools → RepoReader → GithubRepoReader / GitverseRepoReader     чтение репозитория по HTTP
      → AiCheckService.complete (:160) → AiCheck.complete                       UPDATE ai_checks

3. Показ преподавателю:
AssignmentPageController.assignmentPage (:50) → AiCheckService.latestBySubmissionId (:189) → AiCheckView.of(AiCheck)
```

### Куда идут стрелки»

В основном вниз: контроллер и воркер вызывают сервис, сервис вызывает репозиторий, а «низом» здесь служат не только БД, но и внешние API (LLM, GitHub/Gitverse). Есть несоклько исключений:

1. Условно вверх: "RepoReader", адаптер к GitHub/Gitverse, зависит от сервиса "AiSettingsService", чтобы взять токены (RepoReader.java:16–18, :44–48). "AiReviewAgent" и "AiModelFactory" принимают тип "AiProviderConfig", объявленный внутри "AiSettingsService" (AiReviewAgent.java:67, AiModelFactory.java:25).
2. Вбок, между модулями: "aicheck" работает с репозиториями и сущностями модуля "submission" напрямую, в обход "SubmissionService": "SubmissionRepository" и "SubmissionAttachmentRepository" в "AiCheckService" (:29–30), чтение "Submission" и "Assignment" в "claimNext" (:148–153), "AttachmentStorage" в воркере (AiCheckWorker.java:116).

### Как провёл бы границу

1. Настройки передавать вниз как значения, а не через сервис: воркер уже читает настройки ("AiCheckWorker" зависит от "AiSettingsService", :36) и передаёт агенту готовый "AiProviderConfig" (:85). Так же он может передавать и токены: например, record "RepoTokens(github, gitverse)" по цепочке "AiReviewAgent.review" → "RepoTools" → "RepoReader". Адаптеры "GithubRepoReader" и "GitverseRepoReader" уже получают токен параметром (HostRepoReader.java:7–16), поэтому поменяется только "RepoReader": он перестанет зависеть от "AiSettingsService". "AiProviderConfig" вынести из "AiSettingsService" в отдельный record пакета "aicheck", тогда "AiReviewAgent" и "AiModelFactory" будут зависеть от простого значения, а не от класса сервиса. В итоге о настройках знают только воркер и сервис, а адаптеры получают готовые значения.

2. Модуль "submission" отдаёт данные через свой сервис: В "SubmissionService" появляются методы под нужды проверки:
   - "snapshotFor(submissionId)" возвращает данные сдачи для проверки: номер и название задания, описание, ссылки, комментарий, есть ли вложение. Он заменяет чтение сущностей в "claimNext" (:148–153). Тип ответа — свой record модуля "submission", а "ReviewTask" из него собирает уже "aicheck". Иначе появится обратная зависимость "submission → aicheck".
   - "openAttachment(submissionId)" отдаёт файл вложения (путь или поток). Он заменяет "SubmissionAttachmentRepository" в "claimNext" (:153) и "AttachmentStorage" в воркере (AiCheckWorker.java:116).
   - "latestUncheckedIds()" и "latestUncheckedIds(assignmentId)" для бэклога заменяют прямые запросы к "SubmissionRepository" (:85, :92).

### Зaчем границa

Чтобы сбой, отключение или замена ИИ-проверки не мешали приёму работ: сдача сохраняется и без неё.

## Часть 2 - PetClinic через агента

### Агент и модель

Агент: Claude Code, субагент `general-purpose` с чистым контекстом. Модель: Claude Opus 5.5 (`claude-opus-5-5`).

Агенту дали клон https://github.com/spring-projects/spring-petclinic (коммит `500158f` от 29.09.2026) и запретили что-либо запускать или менять. Попросили найти слои, протрассировать «добавление визита» от `VisitController` до записи в БД с номерами строк, ответить на три вопроса из задания и пометить, что проверено по коду, а что предположено.

Время выполнения 3 мин 51 с, 13 вызовов инструментов, входящие 640 836 (из них 571 736 из кэша), исходящие 22 429.

### Результат агента

Пути Java даны от `src/main/java/org/springframework/samples/petclinic/`, шаблонов от `src/main/resources/templates/`, VC = `owner/VisitController.java`. [П] значит проверено чтением кода, [Д] значит допущение о поведении Spring или Hibernate.

**Слои** [П]

- Контроллеры (`@Controller`): `OwnerController`, `PetController`, `VisitController`, `VetController`, `WelcomeController`, `CrashController`.
- Сервисов нет: ни одного `@Service`, `@Transactional` стоит только на двух методах `VetRepository` (:44, :54).
- Репозитории (Spring Data): `OwnerRepository`, `PetTypeRepository`, `VetRepository`. Для `Pet` и `Visit` репозиториев нет, их сохраняет каскад от `Owner`.
- Сущности `@Entity`: `Owner`, `Pet`, `Visit`, `PetType`, `Vet`, `Specialty`; базовые `@MappedSuperclass`: `BaseEntity`, `NamedEntity`, `Person`.
- Вспомогательное для веба: `PetValidator` (подключён только в PetController:94–97), `PetTypeFormatter` (сам ходит в репозиторий, :53), `Vets` (JSON/XML-обёртка). DTO и мапперов нет.

**Трасса «добавление визита»**

```
GET /owners/{o}/pets/{p}/visits/new
LocaleChangeInterceptor (WebConfiguration:57)
→ VC.setAllowedFields:52                          [Д: @InitBinder без имени вызывается для каждого WebDataBinder]
→ VC.loadPetWithVisit:64
    → OwnerRepository.findById:66                 [Д: своя readOnly-транзакция; SELECT owner + EAGER pets (Owner:67)
                                                      + EAGER visits (Pet:56) + type (Pet:52)]
    → Owner.getPet(Integer):129 → BaseEntity.isNew:47
    → model.put("pet"/"owner") VC:75–76
    → new Visit():50 (дата = завтра) → Pet.addVisit:81   несохранённый Visit уже в коллекции сущности
→ VC.minVisitDate:84
→ VC.initNewVisitForm:91 → pets/createOrUpdateVisitForm.html
    visit['new'] → BaseEntity.isNew (:8); th:min=${minVisitDate} (inputField.html:14);
    скрытое поле petId (:38); цикл по pet.visits с фильтром !visit['new'] (:51)


POST на тот же URL
interceptor → VC.setAllowedFields:52 → VC.loadPetWithVisit:64 (вся цепочка выше, снова Pet.addVisit) → VC.minVisitDate:84
→ аргумент @ModelAttribute Owner owner (VC:98)    берётся из модели (:76) и заполняется параметрами запроса
                                                  без @Valid, запрещены только id и *.id (:53)
→ аргумент @Valid Visit visit (VC:98)             тот же объект, что вернула строка :80; привязка date
                                                  (@DateTimeFormat, Visit:40) и description; @NotBlank/@Size (Visit:43–44)
→ VC.processNewVisitForm:98
    → проверка даты :100–101; при ошибках :104–105 тот же шаблон
    → Owner.addVisit:176 → Owner.getPet:129 → Pet.addVisit:81   тот же экземпляр второй раз — no-op
    → OwnerRepository.save:109                    [Д: SimpleJpaRepository.save → isNew=false → em.merge(owner)
                                                      → каскад ALL Owner.pets → Pet.visits → INSERT visits,
                                                      затем UPDATE pet_id (schema.sql:60, pet_id допускает NULL) → commit]
    → flash :110 → redirect:/owners/{ownerId} :111 → OwnerController.showOwner:179
```

**Все ли стрелки идут вниз?** Нет [П]

- Сервисного слоя нет. VC сам держит репозиторий (:45) и сам проводит цепочку загрузить → проверить → изменить → сохранить (:66, :100, :108–109). Правило «визит в будущем» разнесено по трём местам: `Visit()`:50, `VC.minVisitDate`:84 вместе с inputField.html:14 и VC:100.
- Транзакция равна методу репозитория: `findById` (:66) и `save` (:109) идут в разных транзакциях, `open-in-view=false` (application.properties:11), `@Version` нет. [Д: `merge` переписывает весь отсоединённый агрегат, при параллельной правке владельца возможна потеря обновления.]
- Сущности утекают в веб-слой: `Owner`/`Pet`/`Visit` одновременно JPA-сущность, объект формы и модель шаблона (VC:75–76, :80, :98). Следствие: POST на `.../visits/new` с `telephone=…&address=…&pets[0].name=…` перепишет владельца и питомца в обход их валидации (Owner:53–64), а `save` на :109 это сохранит. [П по коду; Д: механика привязки Spring MVC, не запускал.]
- Есть зависимость вверх: сущности несут веб-аннотацию `@DateTimeFormat` (Visit:40, Pet:49) и правила формы.
- Шаблон знает о персистентности: `visit['new']` вызывает `isNew` (:8, :51). Фильтр на :51 нужен только потому, что контроллер меняет сущность уже на GET (VC:79).
- Есть связи вбок: общий фрагмент inputField.html:14 знает про визитный `${minVisitDate}`, хотя тот же фрагмент рисует дату рождения питомца (createOrUpdatePetForm.html:21). `PetTypeFormatter` ходит в репозиторий напрямую (:53).
- Мелочи: скрытое поле `petId` (html:38) ни на что не влияет, такого свойства нет ни у `Visit`, ни у `Owner`. VC:108 повторяет VC:79. Ненайденный владелец даёт `IllegalArgumentException` (VC:67, :72), `@ControllerAdvice` нет. [Д: ответ 500 вместо 404.]
- Сделано правильно: `VisitRepository` нет, потому что `Owner` — корень агрегата, и визит добавляется только через `Owner.addVisit`:176.

**Где бы провёл границу иначе**

Контроллер принимает `record VisitForm(date, description)` с валидацией и `@DateTimeFormat`, а `Owner` в привязку не попадает (только `@PathVariable ownerId/petId`). Сервис `@Transactional VisitService.book(ownerId, petId, form)` загружает `Owner`, проверяет правило даты (или его проверяет сам `Owner.addVisit`), добавляет визит и сохраняет. В шаблон уходит read-model (`PetView` + список визитов) без `isNew`. Граница проходит между контроллером и сервисом, сущности за неё не выходят.

### Зaчем грaница

Чтобы правила и транзакции предметной области жили в одном месте, не зависящем от HTTP и шаблонов, а форма не могла напрямую писать в персистентную модель.